Solutions notebook for the tutor. Released to students only after the lab. Run all cells.

The cells up to section 6 are the student notebook. After them, `Solutions: speedrun` and `Solutions: light` train a sequence of attempts, print the validation accuracy of each and submit each one to its leaderboard, so the board shows scores that go up one after another.

The register cell (section 2) is filled in with the tutor test user (`name.surname@campus.unimib.it`, nickname `...`). Expected run time in Colab: about 5 minutes, including the waits for the 20 s cooldown between submissions.

Every call to the server (register, baseline, attempts) is skipped when the environment variable `AML_SKIP_SUBMIT` is `1`, which is how the file is tested offline. In Colab it is not set.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DBertazioli/aml-labs/blob/main/lab1/2_competition_pytorch.ipynb)

# Lab 1, part 2: competition (PyTorch)

The data is the nested-shells data: 5 classes, 20 columns. Two formats, one live leaderboard.

- **speedrun**: any MLP, **at most 1000 optimizer steps** (a step is one update on one batch). **Batch size is fixed at 128.** The question is how much you can get out of a tiny training budget.
- **light**: any MLP with **at most 50,000 parameters** and **at most 40 epochs**. Batch size is free.

Rules:

- Any MLP: Dense layers, any activation, BatchNorm and Dropout are allowed; no convolutions or other model types. The model must be trained by you in this notebook. Any preprocessing computed from the training data is allowed.
- Honor system: the helpers below count steps, epochs, batch size and parameters from your real training, refuse a submission over budget, and record the numbers with every submission.
- Unlimited submissions (at most one every 20 seconds, counted separately for each leaderboard). The score you see is computed on **half of the test set**. The final ranking uses, for each student, the submission with the best visible score, scored on the **other half**, which you never see.
- Leaderboard: [speedrun and light](https://dbertazioli-aml-labs.hf.space/board?c=speedrun,light).

## 1. Setup

Downloads the class client and the data files. Run it once.

In [ ]:
import os, shutil, urllib.request
REPO = "DBertazioli/aml-labs"
RAW = f"https://raw.githubusercontent.com/{REPO}/main/"
LOCAL = os.environ.get("AML_LOCAL_REPO")  # only used when testing the notebook offline

def fetch(path, name):
    if LOCAL:
        shutil.copy(os.path.join(LOCAL, path), name)
    else:
        urllib.request.urlretrieve(RAW + path, name)

fetch("client/aml_client.py", "aml_client.py")
for name in ["shells_train.npz", "shells_test.npz"]:
    if not os.path.exists(name):
        fetch(f"lab1/competition/{name}", name)
import aml_client as aml

## 2. Register

Register with your campus email and a nickname. We do this together at the start of the competition.

- Your **campus email** (`@campus.unimib.it`) is used **only** to contact competition winners. It is **never shown** to anyone and it is **deleted after the course**.
- Your **nickname appears on the projector** in class, so **do not use your real name**. It has 2-20 characters: letters, digits, `_` or `-`.
- If the nickname is taken, pick another one.

The login is stored in a file. It survives "Restart session"; it is lost when Colab gives you a new runtime (disconnect, timeout, delete runtime). Then run this cell again with the same email and nickname (same spelling).

In [ ]:
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    EMAIL = "name.surname@campus.unimib.it"
    NICKNAME = "..."

    if os.path.exists(aml.TOKEN_FILE):
        print("already registered")
    else:
        if EMAIL == "name.surname@campus.unimib.it" or NICKNAME == "...":
            raise ValueError("Edit EMAIL and NICKNAME in this cell, then run it again.")
        aml.register(EMAIL, NICKNAME)

## 3. Data

Loads the training and test data. Look at the range of each column: some columns are in large units.

In [ ]:
import numpy as np

tr, te = np.load("shells_train.npz"), np.load("shells_test.npz")
X_train, y_train, X_test = tr["X_train"], tr["y_train"], te["X_test"]
K = int(y_train.max()) + 1
print("X_train", X_train.shape, "y_train", y_train.shape, "X_test", X_test.shape)
print("class counts:", np.bincount(y_train))
print("min per column:", np.round(X_train.min(0), 1))
print("max per column:", np.round(X_train.max(0), 1))
print("std per column:", np.round(X_train.std(0), 1))

## 4. The picture

The rule in 3-D: five nested shells, one inside the other, each a ball with a smaller ball on top. The class of a point is the shell it lies in: the innermost shell is class 0, the outermost class 4. Here the shells lie on their side and are cut in half, so you can see the inner shells.

In [ ]:
import plotly.graph_objects as go

colors = ["#0072b2", "#e69f00", "#009e73", "#cc79a7", "#d55e00"]

def rho(theta):   # radius of one shell at polar angle theta, for scale 1 (1 is a sphere)
    dc = 1.35 * np.cos(theta)
    disc = dc**2 - 1.35**2 + 0.6**2
    return np.where(disc >= 0, np.maximum(1, dc + np.sqrt(np.maximum(disc, 0))), 1.0)

def show(traces, eye):
    fig = go.Figure(traces)
    fig.update_layout(height=600, legend=dict(itemsizing="constant"),
                      scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False, aspectmode="data",
                                 camera=dict(eye=dict(x=eye[0], y=eye[1], z=eye[2]), up=dict(x=0, y=0, z=1))))
    fig.show()

# shell axis along x; phi in [pi, 2 pi] keeps the lower half (z <= 0), seen from above
th, ph = np.meshgrid(np.linspace(0, np.pi, 80), np.linspace(np.pi, 2 * np.pi, 60), indexing="ij")
shells = []
for k, c in enumerate(colors):
    r = (1.4 + 0.4 * k) * rho(th)          # outer surface of class k
    shells.append(go.Surface(x=r * np.cos(th), y=r * np.sin(th) * np.cos(ph), z=r * np.sin(th) * np.sin(ph),
                             surfacecolor=np.zeros(th.shape), colorscale=[[0, c], [1, c]], showscale=False,
                             opacity=1, lighting=dict(ambient=0.6, diffuse=0.7, specular=0.1, roughness=0.9),
                             name=f"class {k}", showlegend=True))
show(shells, (-0.8, 1.1, 1.1))

The same rule as points: this is what data from it look like in 3-D, one colour per class (a quarter is cut away so the inner shells show). The competition data uses the same rule in 20 dimensions, which cannot be drawn.

In [ ]:
rng = np.random.default_rng(0)
clouds = []
for k, c in enumerate(colors):
    n = 1870                                                   # about 1400 points are left after the cut
    s = rng.uniform(1 + 0.4 * k, 1.4 + 0.4 * k, n)             # scale inside the band of class k
    cos_t, phi = rng.uniform(-1, 1, n), rng.uniform(0, 2 * np.pi, n)
    sin_t = np.sqrt(1 - cos_t**2)
    r = s * rho(np.arccos(cos_t))
    x, y, z = r * sin_t * np.cos(phi), r * sin_t * np.sin(phi), r * cos_t
    keep = ~((x > 0) & (y > 0))                                # the cut quarter
    clouds.append(go.Scatter3d(x=x[keep], y=y[keep], z=z[keep], mode="markers", marker=dict(size=2, color=c),
                               name=f"class {k}"))
show(clouds, (1.5, 1.5, 1.0))

## 5. Budget helpers

Training goes through the provided function (`fit` in Keras, `train` in PyTorch). It counts the steps, epochs, samples and batch size of the real training and stores them on the model. Running a training cell again keeps counting on the same model: build a new model to start over. `train` takes an optional learning-rate scheduler (`sched=...`): it is stepped after every optimizer step. `submit` computes the number of parameters, checks the budget of the format, predicts the classes of `X_test` and sends the predictions. If you preprocess the inputs for training, pass the same function as `preprocess` so the test inputs are transformed the same way.

The same function plots the training loss after every epoch, so the plot updates while training runs (`LivePlot`, defined in the cell below). *Further reading: [IPython.display.clear_output](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.clear_output) (the redraw)*

In [ ]:
import torch, torch.nn as nn

FORMATS = {"speedrun": "at most 1000 steps, batch size 128",
           "light": "at most 50,000 parameters, at most 40 epochs"}

import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

class LivePlot:
    """Keeps the per-epoch values and redraws the curves while training runs.
    update(logs) adds one epoch (redraw at most every `every` seconds); draw() shows the current state."""
    def __init__(self, every=0.5):
        self.h, self.every, self.t, self.title = {}, every, time.time(), ""   # h: (run, name) -> values per epoch
    def update(self, logs, run=""):                   # run: curve name when one plot shows several runs
        for k, v in logs.items():
            self.h.setdefault((run, k), []).append(float(v))
        n = len(self.h[(run, k)])
        self.title = f"{run + ': ' if run else ''}epoch {n} | " + " | ".join(f"{k} {float(v):.3f}" for k, v in logs.items())
        if time.time() - self.t > self.every:
            self.draw()
    def draw(self):
        if not self.h:
            return
        panels = [p for p in ("loss", "accuracy") if any(p in k for _, k in self.h)]
        fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 3), squeeze=False)
        for ax, p in zip(axes[0], panels):
            for (run, k), v in self.h.items():
                if p in k:
                    ax.plot(range(1, len(v) + 1), v, label=run or k)
            ax.set_xlabel("epoch"); ax.set_title(p); ax.legend()
        fig.suptitle(self.title); fig.tight_layout()
        clear_output(wait=True); display(fig); plt.close(fig)
        self.t = time.time()

def count_params(model):
    return sum(p.numel() for p in model.parameters())   # all parameters, same count as in the Keras version

def train(model, opt, X, y, epochs=1, batch_size=128, max_steps=None, sched=None):
    """Minibatch training loop. Stops after `epochs` epochs or when the model has done `max_steps` steps in total.
    Steps, epochs, samples seen and batch size are added to the model (cumulative over calls).
    sched: optional learning-rate scheduler, stepped after every optimizer step."""
    if not hasattr(model, "aml_steps"):
        model.aml_steps = model.aml_epochs = model.aml_samples = 0
        model.aml_bs_min = model.aml_bs_max = batch_size
    model.aml_bs_min, model.aml_bs_max = min(model.aml_bs_min, batch_size), max(model.aml_bs_max, batch_size)
    model.aml_absmax = float(np.abs(np.asarray(X)).max())
    X, y = torch.tensor(X, dtype=torch.float32), torch.tensor(y)
    model.train()
    live = LivePlot()
    for _ in range(epochs):
        if max_steps and model.aml_steps >= max_steps:
            break
        model.aml_epochs += 1
        perm = torch.randperm(len(X))
        losses = []
        for i in range(0, len(X) - batch_size + 1, batch_size):
            b = perm[i:i + batch_size]
            loss = nn.functional.cross_entropy(model(X[b]), y[b])
            opt.zero_grad(); loss.backward(); opt.step()
            if sched is not None:
                sched.step()
            losses.append(loss.item())
            model.aml_steps += 1
            model.aml_samples += batch_size
            if max_steps and model.aml_steps >= max_steps:
                break
        live.update({"loss": np.mean(losses)})
    live.draw()
    print("total so far: steps", model.aml_steps, "| epochs", model.aml_epochs,
          "| batch size", model.aml_bs_max, "| params", count_params(model))

@torch.no_grad()
def predict(model, X):
    model.eval()
    return model(torch.tensor(X, dtype=torch.float32)).argmax(1).numpy()

def submit(fmt, model, preprocess=lambda X: X):
    """fmt: "speedrun" or "light". Steps, epochs and batch size come from your training with train()."""
    if fmt not in FORMATS:
        raise ValueError('fmt must be "speedrun" or "light"')
    if not hasattr(model, "aml_steps"):
        raise ValueError("this model was not trained with train(...): train it with the provided train function")
    steps, epochs, params = model.aml_steps, model.aml_epochs, count_params(model)
    if fmt == "speedrun" and steps > 1000:
        raise ValueError(f"speedrun allows at most 1000 steps, this model has {steps} (steps add up over repeated training: build a new model)")
    if fmt == "speedrun" and (model.aml_bs_min, model.aml_bs_max) != (128, 128):
        raise ValueError(f"speedrun needs batch size 128, you used {model.aml_bs_min}-{model.aml_bs_max}")
    if fmt == "light" and params > 50_000:
        raise ValueError(f"light allows at most 50,000 parameters, your model has {params}")
    if fmt == "light" and (epochs > 40 or model.aml_samples > 40 * len(X_train)):
        raise ValueError(f"light allows at most 40 epochs, this model has {epochs} (epochs add up over repeated training: build a new model)")
    r = np.abs(preprocess(X_train)).max() / model.aml_absmax
    if r < 0.5 or r > 2:
        raise ValueError("The model was trained on preprocessed inputs: pass the same preprocess to submit(..., preprocess=...)")
    preds = predict(model, preprocess(X_test))
    return aml.compete(fmt, preds, steps=steps, epochs=epochs, params=params,
                       batch_size=model.aml_bs_max, framework="pytorch")

**Tutor version only: cooldown.** The server accepts one submission every 20 seconds per leaderboard. `wait_cooldown(fmt)` sleeps the time left since the previous submit to that leaderboard (plus 1 second), so Run all never gets a rejection. It does nothing when `AML_SKIP_SUBMIT` is `1`.

In [ ]:
import time

last = {}   # time of the last submit per leaderboard

def wait_cooldown(fmt, gap=21):   # 20 s limit plus 1 s margin
    if os.environ.get("AML_SKIP_SUBMIT") != "1":
        time.sleep(max(0, last.get(fmt, 0) + gap - time.time()))
    last[fmt] = time.time()

## 6. Baseline

A small MLP (2 hidden layers of 64 ReLU units), plain SGD with learning rate 0.01, raw inputs. This is what you have to beat. The same model also fits the `light` budget (about 5.8k parameters, 7 epochs), so it is submitted to both.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(X_train.shape[1], 64), nn.ReLU(),
                      nn.Linear(64, 64), nn.ReLU(),
                      nn.Linear(64, K))
opt = torch.optim.SGD(model.parameters(), lr=0.01)
train(model, opt, X_train, y_train, epochs=100, batch_size=128, max_steps=1000)   # speedrun: stops at 1000 steps
print("train accuracy:", (predict(model, X_train) == y_train).mean())
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("speedrun")
    submit("speedrun", model)
    wait_cooldown("light")
    submit("light", model)

**Solutions.** Every attempt below trains on the first 16,000 rows, prints its validation accuracy (last 4,000 rows) and submits that same model. Nothing is retrained on all 20,000 rows. Before each submit the notebook waits for the 20 s cooldown of that leaderboard.

## Solutions: speedrun

Improvement attempts for the speedrun format: at most 1000 steps at batch size 128, counted by `train`. Every attempt trains on the first 16,000 rows, is scored on a held-out validation split of the **training** data (the last 4,000 rows) and is then submitted to the `speedrun` leaderboard. The test set is used only for the predictions that go to the server. Learning rates were picked by trying a few values on the same validation split, so the numbers are slightly optimistic. Each attempt is one run (seed 0): differences below about 1 point are within the noise of a 4,000-row validation split.

In [ ]:
import math

n_val = 4000     # validation: the last 4,000 rows of the training data; the attempts train on the other rows
X_fit, y_fit, X_val, y_val = X_train[:-n_val], y_train[:-n_val], X_train[-n_val:], y_train[-n_val:]
mean, std = X_fit.mean(0), X_fit.std(0)
standardize = lambda X: ((X - mean) / std).astype("float32")
R = {"speedrun": {}, "light": {}}     # results per attempt

def mlp(n_in, width, depth, bn=False, dropout=0.0):
    net, p = [], n_in
    for _ in range(depth):
        net += [nn.Linear(p, width)] + ([nn.BatchNorm1d(width)] if bn else []) + [nn.ReLU()]
        net += [nn.Dropout(dropout)] if dropout else []
        p = width
    return nn.Sequential(*net, nn.Linear(p, K))

def run(part, name, preprocess=lambda X: X, width=64, depth=2, bn=False, dropout=0.0, optimizer="sgd", lr=0.01,
        cosine=False, epochs=100, max_steps=None, seed=0):
    torch.manual_seed(seed)
    model = mlp(preprocess(X_fit[:1]).shape[1], width, depth, bn, dropout)
    steps = max_steps or epochs * (len(X_fit) // 128)
    warm = steps // 20
    opt = {"sgd": lambda: torch.optim.SGD(model.parameters(), lr=lr), "adam": lambda: torch.optim.Adam(model.parameters(), lr=lr),
           "adamw": lambda: torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)}[optimizer]()
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: min(1, (t + 1) / warm) * 0.5 * (1 + math.cos(math.pi * t / steps))) if cosine else None
    train(model, opt, preprocess(X_fit), y_fit, epochs=epochs, batch_size=128, max_steps=max_steps, sched=sched)
    acc = float((predict(model, preprocess(X_val)) == y_val).mean())
    acc_train = float((predict(model, preprocess(X_fit[:n_val])) == y_fit[:n_val]).mean())
    steps, params = model.aml_steps, count_params(model)
    assert (part != "speedrun" or (steps <= 1000 and model.aml_bs_max == 128)) and (part != "light" or (params <= 50_000 and model.aml_epochs <= 40)), "over budget"
    R[part][name] = dict(acc=acc, steps=steps, epochs=model.aml_epochs, params=params, model=model, pre=preprocess)
    print(f"{name}: validation accuracy {acc:.4f} (training accuracy {acc_train:.4f}) | steps {steps} | epochs {model.aml_epochs} | params {params:,}")
    return R[part][name]

from IPython.display import Markdown, display

def table(part):
    use = (lambda r: f"{r['steps']} of 1000 steps") if part == "speedrun" else (
          lambda r: f"{r['params']:,} of 50,000 parameters, {r['epochs']} of 40 epochs")
    rows = ["| attempt | validation accuracy | budget use | submitted |", "|---|---|---|---|"]
    rows += [f"| {n} | {r['acc']:.4f} | {use(r)} | {'yes' if r.get('submitted') else 'no'} |" for n, r in R[part].items()]
    display(Markdown("\n".join(rows)))

**Attempt 0: the baseline as given**, retrained on the 16,000 rows (SGD, learning rate 0.01, raw inputs, 2 x 64 ReLU).

In [ ]:
r = run("speedrun", "0 baseline (as given)", epochs=100, max_steps=1000)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("speedrun")
    submit("speedrun", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Attempt 1: standardize the inputs.** A few columns have a range about 100 times larger than the others (section 3), so they dominate the gradient and the learning rate has to stay small. After standardizing (mean and standard deviation of the training rows) all columns have the same scale, and plain SGD works with a 10 times larger learning rate (0.1).

In [ ]:
r = run("speedrun", "1 standardize + SGD", standardize, optimizer="sgd", lr=0.1, epochs=100, max_steps=1000)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("speedrun")
    submit("speedrun", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Attempt 2: Adam.** Adam divides each parameter's step by the size of its recent gradients, so the columns and layers that need a different step size get one. Learning rate 0.01.

In [ ]:
r = run("speedrun", "2 Adam", standardize, optimizer="adam", lr=0.01, epochs=100, max_steps=1000)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("speedrun")
    submit("speedrun", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Attempt 3: warmup and cosine decay.** With 1000 steps the step size matters at both ends: a short linear warmup (5% of the steps), a higher peak learning rate (0.03) and a decay to zero at the last step. AdamW (Adam with weight decay 1e-4).

In [ ]:
r = run("speedrun", "3 AdamW + warmup + cosine", standardize, optimizer="adamw", lr=0.03, cosine=True, epochs=100, max_steps=1000)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("speedrun")
    submit("speedrun", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Attempt 4: wider network with BatchNorm.** Three hidden layers of 256 units, BatchNorm before each ReLU. BatchNorm keeps the activations well scaled, so the high peak learning rate keeps working and the model fits more in the same 1000 steps.

In [ ]:
r = run("speedrun", "4 3 x 256 + BatchNorm", standardize, width=256, depth=3, bn=True, optimizer="adamw", lr=0.03, cosine=True, epochs=100, max_steps=1000)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("speedrun")
    submit("speedrun", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Speedrun summary.** Every attempt used exactly 1000 steps at batch size 128.

In [ ]:
table("speedrun")
best = max(R["speedrun"], key=lambda n: R["speedrun"][n]["acc"])
print("best on validation:", best)

## Solutions: light

Improvement attempts for the light format: at most 50,000 parameters and at most 40 epochs (batch size 128). Same split as above: the attempts train for 40 epochs on the first 16,000 rows (5,000 steps), are scored on the last 4,000 and are then submitted to the `light` leaderboard. The dropout rate was also picked on this split. The parameter count and the epochs come from the notebook's own helpers.

**Attempt 0: the baseline as given** (about 5.8k parameters, 8 epochs on the 16,000 rows), which also fits the light budget.

In [ ]:
r = run("light", "0 baseline (as given)", epochs=100, max_steps=1000)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("light")
    submit("light", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Attempt 1: wider and deeper, within the budget.** Standardized inputs, three hidden layers of 128 units (about 36,000 parameters), AdamW with a constant learning rate (3e-3), 40 epochs.

In [ ]:
r = run("light", "1 3 x 128 + AdamW", standardize, width=128, depth=3, optimizer="adamw", lr=3e-3, epochs=40)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("light")
    submit("light", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Attempt 2: warmup and cosine decay.** Same network. The 5,000 steps allow a high peak learning rate (0.03) that decays to zero. Compare with the constant learning rate of attempt 1.

In [ ]:
r = run("light", "2 + warmup + cosine", standardize, width=128, depth=3, optimizer="adamw", lr=0.03, cosine=True, epochs=40)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("light")
    submit("light", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Attempt 3: a feature computed from the training data.** The Euclidean norm of the standardized input is added as one extra column (printed: its mean per class, which grows with the class). Same network as attempt 2.

In [ ]:
add_norm = lambda X: np.c_[standardize(X), np.linalg.norm(standardize(X), axis=1)].astype("float32")
print("mean norm per class:", np.round([np.linalg.norm(standardize(X_fit[y_fit == k]), axis=1).mean() for k in range(K)], 2))
r = run("light", "3 + norm feature", add_norm, width=128, depth=3, optimizer="adamw", lr=0.03, cosine=True, epochs=40)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("light")
    submit("light", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Attempt 4: BatchNorm and Dropout.** The training accuracy is far above the validation accuracy: the model overfits the 16,000 rows. Dropout 0.3 after each hidden layer, with BatchNorm to keep the activations well scaled while units are dropped. Three layers of 148 units (about 49,000 parameters, close to the limit), same schedule and inputs as attempt 3. The two are one attempt on purpose: in quick tests on this data BatchNorm alone moved the validation accuracy by less than 1 point and Dropout alone lowered it. The gain over attempt 3 is about 1 point, close to the noise of the validation split.

In [ ]:
r = run("light", "4 + BatchNorm + Dropout (3 x 148)", add_norm, width=148, depth=3, bn=True, dropout=0.3, optimizer="adamw", lr=0.03, cosine=True, epochs=40)
if os.environ.get("AML_SKIP_SUBMIT") != "1":
    wait_cooldown("light")
    submit("light", r["model"], preprocess=r["pre"])   # the model trained on the 16,000 rows
    r["submitted"] = True

**Light summary.**

In [ ]:
table("light")
best = max(R["light"], key=lambda n: R["light"][n]["acc"])
print("best on validation:", best)